# Baseline Models: XGBoost · Random Forest · LightGBM

| | |
|---|---|
| **Train** | `train_after_smote.csv` — 333,650 mẫu (đã SMOTE, cân bằng 5 nhóm) |
| **Test**  | `test_set.csv` — 18,658 mẫu (phân phối thực tế) |
| **Target** | `NHOMNOMOI` (nhóm 1–5) |
| **Đánh giá** | `classification_report` + bảng so sánh tổng hợp |

In [ ]:
# pip install xgboost lightgbm scikit-learn pandas numpy
import pandas as pd
import numpy as np
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier



✅ Thư viện load thành công!


---
## 1. Load dữ liệu

In [3]:
train = pd.read_csv('train_after_smote.csv')
test  = pd.read_csv('test_set.csv')

X_train = train.drop(columns=['NHOMNOMOI'])
y_train = train['NHOMNOMOI']
X_test  = test.drop(columns=['NHOMNOMOI'])
y_test  = test['NHOMNOMOI']

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print('\nPhân phối Train (sau SMOTE):')
print(y_train.value_counts().sort_index())
print('\nPhân phối Test (thực tế):')
print(y_test.value_counts().sort_index())

Train : (333650, 12)  |  Test : (18658, 12)

Phân phối Train (sau SMOTE):
NHOMNOMOI
1    66730
2    66730
3    66730
4    66730
5    66730
Name: count, dtype: int64

Phân phối Test (thực tế):
NHOMNOMOI
1    16683
2      773
3      257
4      267
5      678
Name: count, dtype: int64


---
## 2. Tính class_weight từ tập Test (phân phối thực tế)

Dù train đã cân bằng qua SMOTE, một số model vẫn hỗ trợ `class_weight` / `scale_pos_weight`  
để nhấn mạnh các nhóm thiểu số khi đánh giá trên tập test thực tế.

In [ ]:
classes = np.array(sorted(y_train.unique()))   # [1, 2, 3, 4, 5]

weights_arr = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights_arr))

print("class_weight (balanced):")
for cls, w in class_weight_dict.items():
    print(f"  Nhom {cls}: {w:.4f}")

class_weight (balanced):
  Nhom 1: 1.0000
  Nhom 2: 1.0000
  Nhom 3: 1.0000
  Nhom 4: 1.0000
  Nhom 5: 1.0000


---
## 3. Hàm tiện ích

In [10]:
LABEL_NAMES = [f'Nhom {i}' for i in range(1, 6)]

def run_baseline(name, model, label_offset=0):
    """
    Train model trên X_train/y_train, đánh giá trên X_test/y_test.
    label_offset: cộng vào y_pred sau predict (dùng cho XGBoost nhãn 0-4).
    """
    print(f'\n{"="*62}')
    print(f'  {name}')
    print(f'{"="*62}')

    y_tr = y_train - label_offset   # shift nhãn nếu cần

    t0 = time.time()
    model.fit(X_train, y_tr)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test) + label_offset

    print(f'Training time : {elapsed:.1f}s')
    print()
    print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))
    return model

---
## 4. Random Forest Baseline

| Tham số điều chỉnh | Giá trị | Mục đích |
|---|---|---|
| `class_weight` | `balanced` | Tự động điều chỉnh trọng số theo tần suất nhãn |
| `n_estimators` | 100 | Số cây (baseline) |
| `max_depth` | None | Cây mọc đầy đủ |
| `min_samples_leaf` | 1 | Không hạn chế lá |

In [6]:
rf_model = run_baseline(
    'RANDOM FOREST  (Baseline · class_weight=balanced)',
    RandomForestClassifier(
        n_estimators   = 100,
        max_depth      = None,
        min_samples_leaf = 1,
        class_weight   = 'balanced',   # <-- điều chỉnh trọng số
        random_state   = 42,
        n_jobs         = -1
    )
)


  RANDOM FOREST  (Baseline · class_weight=balanced)
Training time : 10.0s

              precision    recall  f1-score   support

      Nhom 1       0.93      0.78      0.85     16683
      Nhom 2       0.05      0.13      0.08       773
      Nhom 3       0.02      0.08      0.03       257
      Nhom 4       0.06      0.21      0.09       267
      Nhom 5       0.66      0.62      0.64       678

    accuracy                           0.73     18658
   macro avg       0.34      0.36      0.34     18658
weighted avg       0.86      0.73      0.79     18658



---
## 5. XGBoost Baseline

| Tham số điều chỉnh | Giá trị | Mục đích |
|---|---|---|
| `sample_weight` | `class_weight_dict` | Truyền trọng số mẫu khi fit |
| `scale_pos_weight` | — | Dùng cho binary; multiclass dùng `sample_weight` |
| `n_estimators` | 100 | Số cây |
| `max_depth` | 6 | Độ sâu cây |
| `learning_rate` | 0.1 | Tốc độ học |

> ⚠️ XGBoost yêu cầu nhãn bắt đầu từ 0 → dùng `y_train - 1` khi fit, predict xong cộng lại `+1`.

In [7]:
# Tạo mảng sample_weight cho từng dòng train
sample_weights_xgb = y_train.map(class_weight_dict).values

xgb_model = XGBClassifier(
    n_estimators  = 100,
    max_depth     = 6,
    learning_rate = 0.1,
    eval_metric   = 'mlogloss',
    random_state  = 42,
    n_jobs        = -1
)

print(f'\n{"="*62}')
print(f'  XGBOOST  (Baseline · sample_weight=class_weight)')
print(f'{"="*62}')

t0 = time.time()
xgb_model.fit(
    X_train, y_train - 1,              # nhãn 0-4
    sample_weight=sample_weights_xgb   # <-- điều chỉnh trọng số
)
print(f'Training time : {time.time()-t0:.1f}s\n')

y_pred_xgb = xgb_model.predict(X_test) + 1   # +1 để về nhãn 1-5
print(classification_report(y_test, y_pred_xgb, target_names=LABEL_NAMES))


  XGBOOST  (Baseline · sample_weight=class_weight)
Training time : 4.4s

              precision    recall  f1-score   support

      Nhom 1       0.94      0.74      0.83     16683
      Nhom 2       0.07      0.16      0.09       773
      Nhom 3       0.03      0.17      0.04       257
      Nhom 4       0.06      0.32      0.10       267
      Nhom 5       0.70      0.60      0.65       678

    accuracy                           0.70     18658
   macro avg       0.36      0.40      0.34     18658
weighted avg       0.87      0.70      0.77     18658



---
## 6. LightGBM Baseline

| Tham số điều chỉnh | Giá trị | Mục đích |
|---|---|---|
| `class_weight` | `balanced` | Tự động điều chỉnh trọng số |
| `n_estimators` | 100 | Số cây |
| `learning_rate` | 0.1 | Tốc độ học |
| `num_leaves` | 31 | Số lá mặc định |

In [8]:
lgbm_model = run_baseline(
    'LIGHTGBM  (Baseline · class_weight=balanced)',
    LGBMClassifier(
        n_estimators  = 100,
        learning_rate = 0.1,
        num_leaves    = 31,
        class_weight  = 'balanced',   # <-- điều chỉnh trọng số
        random_state  = 42,
        n_jobs        = -1,
        verbose       = -1
    )
)


  LIGHTGBM  (Baseline · class_weight=balanced)
Training time : 3.0s

              precision    recall  f1-score   support

      Nhom 1       0.94      0.79      0.86     16683
      Nhom 2       0.07      0.14      0.09       773
      Nhom 3       0.03      0.16      0.05       257
      Nhom 4       0.07      0.28      0.11       267
      Nhom 5       0.74      0.63      0.68       678

    accuracy                           0.74     18658
   macro avg       0.37      0.40      0.36     18658
weighted avg       0.87      0.74      0.80     18658



---
## 7. Bảng so sánh tổng hợp

In [11]:
results = []

models_info = [
    ('Random Forest', rf_model,   0),
    ('XGBoost',       xgb_model,  1),   # +1 về nhãn gốc
    ('LightGBM',      lgbm_model, 0),
]

for name, model, offset in models_info:
    y_pred = model.predict(X_test) + offset
    results.append({
        'Model':                name,
        'Accuracy':             round(accuracy_score(y_test, y_pred), 4),
        'Precision (weighted)': round(precision_score(y_test, y_pred, average='weighted'), 4),
        'Recall (weighted)':    round(recall_score(y_test, y_pred, average='weighted'), 4),
        'F1 (weighted)':        round(f1_score(y_test, y_pred, average='weighted'), 4),
        'F1 (macro)':           round(f1_score(y_test, y_pred, average='macro'), 4),
    })

df_results = pd.DataFrame(results).set_index('Model')
print('\nSo sánh Baseline (class_weight=balanced):')
df_results


So sánh Baseline (class_weight=balanced):


,Accuracy,Precision (weighted),Recall (weighted),F1 (weighted),F1 (macro)
Model,,,,,
Random Forest,0.7325,0.8567,0.7325,0.7876,0.3375
XGBoost,0.6987,0.8704,0.6987,0.7712,0.3428
LightGBM,0.7395,0.8694,0.7395,0.7965,0.3566
